### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [34]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

import numpy as np


In [35]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [36]:
# Deprecated 
# llm = ChatGroq(model="gemma2-9b-it", temperature=0)
# llm


In [37]:
# Other was of initializing the Groq LLM

llm = init_chat_model("groq:meta-llama/llama-4-maverick-17b-128e-instruct", temperature=0)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000253042EE750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000025303F81790>, model_name='meta-llama/llama-4-maverick-17b-128e-instruct', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [38]:
response = llm.invoke("What is RAG?")
response

AIMessage(content="RAG stands for Retrieval-Augmented Generation. It's a technique used in natural language processing (NLP) and artificial intelligence (AI) to improve the performance of language models, particularly in tasks that require generating text based on a given prompt or question.\n\nThe RAG approach involves two main components:\n\n1. **Retrieval**: The model retrieves relevant information from a knowledge base or a database, often using a dense retriever or a sparse retriever. This step helps to gather relevant context or evidence that can inform the generation process.\n2. **Generation**: The retrieved information is then used to augment the generation process. The model generates text based on the input prompt or question, incorporating the retrieved information to produce a more accurate and informed response.\n\nThe RAG technique has several benefits, including:\n\n* **Improved accuracy**: By retrieving relevant information, the model can generate more accurate and inf

In [39]:
#  If your ChromaDB was created with persistence, you can connect to it like this

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

vector_db_path = os.path.abspath(os.path.join(os.getcwd(), "..", "VectorDBs","ChromaDB"))
collection = "kcj-langchain-rag-chromadb"
print(f"Changing working directory to: {vector_db_path}")

vectordb = Chroma(
    persist_directory=vector_db_path,
    embedding_function=embeddings,
    collection_name= collection
)

retriever = vectordb.as_retriever()

Changing working directory to: d:\krishna-codejournal\kcj-langchain-rag-starter\src\LangChain\VectorDBs\ChromaDB


In [40]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_core.prompts import ChatPromptTemplate   

from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [41]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [42]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002530434E480>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChun

In [43]:
# Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# Create conversational RAG chain
conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    question_answer_chain
)
print("Conversational RAG chain created!")

Conversational RAG chain created!


In [44]:
chat_history=[]

# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to improve their performance on a task over time, based on experience. Machine learning is used in applications such as image recognition, natural language processing, and recommender systems.


In [45]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [46]:
chat_history

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to improve their performance on a task over time, based on experience. Machine learning is used in applications such as image recognition, natural language processing, and recommender systems.', additional_kwargs={}, response_metadata={})]

In [49]:
## Follow up question
# Follow-up question
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})
result2

{'chat_history': [HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to improve their performance on a task over time, based on experience. Machine learning is used in applications such as image recognition, natural language processing, and recommender systems.', additional_kwargs={}, response_metadata={})],
 'input': 'What are its main types?',
 'context': [],
 'answer': 'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning involves training models on labeled data, while unsupervised learning involves training on unlabeled data to discover patterns. Reinforcement learning involves training models to make decisions based on rewards or penalties.'}

In [50]:
result2['answer']

'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning involves training models on labeled data, while unsupervised learning involves training on unlabeled data to discover patterns. Reinforcement learning involves training models to make decisions based on rewards or penalties.'